# Overhead Lines and Underground Cables


All overhead and underground AC lines are represented as `ACLineSegment` objects. Like all `ConductingEquipment`, all lines and cables are defined with two Terminal objects with `ACDCTerminal:sequenceNumber` set to 1 and 2 to represent the two ends of the line, rather than specifying a from-bus and to-bus. There are four different ways to specify `ACLineSegment` impedances and admittances. The first two use positive and zero sequence values; the third specifies the lower triangular R, X, and B values for each conductor phase; the fourth uses the conductor material, geometry, and spacing. In all cases, the `Conductor:length` attribute is required. A combination of all four methods may be used in a single model to define the network. 



The first way, depicted in the figure below, is to specify the individual positive sequence and zero sequence R, X, and B values as `ACLineSegment` attributes, in a manner similar to the method used to define line impedance in many bus-branch transmission analysis tools, such as PSSE. A power system application importing the CIM model would directly use the specified attributes to run a power flow solution. The `PerLengthImpedance` attribute is left as null. The classes and attributes used in this method are shown in the figure below.


In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [2]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor])
Mermaid(diagram_text)


The second way, displayed in the figure below, is to specify the positive and zero sequence impedance and admittance values on a per-unit-length basis as attributes of the `PerLengthSequenceImpedance` class. A power system application importing the CIM model would multiply the specified R, X, and B values by the `Conductor:length` to determine the overall line impedance. When using this method, all the `ACLineSegment` attributes should be null.


In [3]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance])
Mermaid(diagram_text)


The third way to specify line parameters, shown in the figure below, is to define R, X, and B values for each conductor phase. This method is more useful for distribution networks for which an unbalanced power flow solution is needed. The impedance and admittance values are specified as attributes of the `PhaseImpedanceData` class, which inherits from `PerLengthPhaseImpedance`. Again, all the ACLineSegment attributes are left null. Only conductorCount from 1 to 3 is supported, and there will be 1, 3 or 6 reverse-associated `PhaseImpedanceData` instances that define the lower triangle of the Z and Y matrices per unit length. The row and column attributes must agree with `ACLineSegmentPhase:sequenceNumber`.


In [4]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance,cim.ACLineSegmentPhase,cim.SinglePhaseKind,cim.PhaseImpedanceData,cim.PerLengthPhaseImpedance])
Mermaid(diagram_text)

<p style="text-align: justify;">
The fourth way, shown in the figure below, is to specify wire/cable geometry and spacing data instead of impedances. Unlike the previous three methods (where the impedance values were specified through the 61970 Wires package), all attributes of the line are specified through physical attributes as part of the 61968 AssetInfo package.  Conductor spacing is specified by association to *WireSpacingInfo* and *WirePosition*. Conductor geometry is specified through the attributes of *WireInfo*. Cables are specified through the CableInfo class and associated *ConcentricNeutralCableInfo* and *TapeShieldCableInfo* classes.
</p>


If there are *ACLineSegmentPhase* instances reverse-associated to the *ACLineSegment*, then per-phase modeling applies. There are several use cases for the *ACLineSegmentPhase* class:
1)	single-phase, two-phase, or three-phase unbalanced primary lines
2)	low-voltage secondary lines using phases s1 and s2
3)	associated WireInfo data where the WireSpacingInfo association exists
4)	assign specific phases to the matrix rows and columns in PerLengthPhaseImpedance. 


<p style="text-align: justify;">
It is the application’s responsibility to propagate phasing through terminals to other components, and to identify any miswiring. It is also the application’s responsibility to calculate the impedance and admittance values for the electrical network from the conductor geometry and spacing. The associations between all the AssetInfo classes described above are shown in the figure below.
</p>

In [5]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance,cim.ACLineSegmentPhase,cim.SinglePhaseKind,cim.PhaseImpedanceData,cim.PerLengthPhaseImpedance,cim.PerLengthLineParameter,cim.WireAssemblyInfo,cim.WireSpacingInfo,cim.WireInfo,cim.ConcentricNeutralCableInfo,cim.TapeShieldCableInfo,cim.CableInfo,cim.WireMaterialKind,cim.CableConstructionKind,cim.CableShieldMaterialKind,cim.WireInsulationKind,cim.CableOuterJacketKind])
Mermaid(diagram_text)

----

Some examples related to overhead lines and underground cables are discussed below.


In [6]:
from cimgraph.models import FeederModel
from cimgraph.databases import ConnectionParameters, XMLFile
import cimgraph.data_profile.cimhub_2023 as cim

In [7]:
params = ConnectionParameters(filename='../sample_models/ieee13.xml',
                              cim_profile='cimhub_2023',
                              iec61970_301=8) # file path
file = XMLFile(params) # file read connection
network = FeederModel(container=cim.Feeder(),connection=file) # create feeder model

ConnectionParameters class is deprecated and will be deleted in a future release
Set environment variables for required authentication


Example 1: What are the phases of Line named 632645?


In [8]:
result = []
name = '632645'

# Check that the graph contains ACLineSegment
if cim.ACLineSegment in network.graph:
    #Traverse through all ACLineSegment instances in the network graph
    for line in network.graph[cim.ACLineSegment].values():
        # Check if the line's name matches the specified name '632645'
        if name in line.name:
            # Loop through all ACLineSegmentPhases associated with the ACLineSegment
            for phases in line.ACLineSegmentPhases:
                # Append the phase to the result list
                result.append(str(phases.phase))
            break # break after we find line with correct name

# Output the result which contains the phases of the line named '632645'
print(result)

['SinglePhaseKind.C', 'SinglePhaseKind.B']


In [9]:
diagram_text = utils.get_mermaid_path(line,['ACLineSegmentPhases','[0]','phase'])
diagram_text = utils.add_mermaid_path(line,'ACLineSegmentPhases[1].phase', diagram_text)
Mermaid(diagram_text)

Example 2: Find the length of the line named 632645?

In [10]:
results = []
name = '632645'

# Initialize the result variable to store the length of the line
result = None

# Check that the graph contains ACLineSegment
if cim.ACLineSegment in network.graph:
    # Traverse through all ACLineSegment instances in the network graph
    for line in network.graph[cim.ACLineSegment].values():
        # Check if the line's name matches the specified name '632645'
        if name in line.name:
            # Get the length of the ACLineSegment
            results = line.length
            break

# Output the result which contains the length of the line named '632645'
print(results)

152.4


In [11]:
diagram_text = utils.get_mermaid_path(line,'length')
Mermaid(diagram_text)

Example 3: What is the nominal voltage of the line named 632645?

In [12]:
results = []
name = '632645'

# Check that the graph contains ACLineSegment
if cim.ACLineSegment in network.graph:
    # Iterate through all ACLineSegment instances in the network graph
    for line in network.graph[cim.ACLineSegment].values():
        # Check if the line's name contains the target name
        if name in line.name:
            # Get the BaseVoltage object
            base_voltage = line.BaseVoltage
            # Get the nominalVoltage attribute:
            results.append(base_voltage.nominalVoltage)
            break  # Exit loop since we found the required line

print(results)

[4160.0]


In [13]:
diagram_text = utils.get_mermaid_path(line,'BaseVoltage')
diagram_text = utils.add_mermaid_path(base_voltage, 'nominalVoltage', diagram_text)
Mermaid(diagram_text)

In [14]:
# diagram_text = utils.get_mermaid_path(cim.ACLineSegment,'BaseVoltage')
# Mermaid(diagram_text)

Example 4: Find how many lines use overhead wire acsr_4/0?

In [15]:
results = []
name = 'acsr_4/0'
lines = []
# Check that the graph contains any OverheadWireInfo
if cim.OverheadWireInfo in network.graph:
    # Traverse through all OverheadWireInfo instances in the network graph
    for wire_info in network.graph[cim.OverheadWireInfo].values():
        # Check if the overhead wire name matches the specified name 'acsr_4/0'
        if wire_info.name == name:
            # Loop through all ACLineSegmentPhases associated with the OverheadWireInfo
            for ac_line_segment_phase in wire_info.ACLineSegmentPhases:
                # Get the ACLineSegment associated with the ACLineSegmentPhase
                ac_line_segment = ac_line_segment_phase.ACLineSegment
                # Check if the ACLineSegment name is not already in the results list
                if ac_line_segment.mRID not in lines:
                    # Add the ACLineSegment name to the results list
                    lines.append(ac_line_segment.mRID)

# Count the total number of unique lines
results = len(lines)
# Output the total number of unique lines using the specified overhead wire construction type
print(results)

0


Example 5: Find the names of buses connected to line named 632645?

In [16]:
results = []
name = '632645'

# Traverse through all ACLineSegment instances in the network graph
for line in network.graph[cim.ACLineSegment].values():
    # Check if the line's name matches the specified name 
    if name in line.name:
        # Loop through all Terminals associated with the ACLineSegment
        for terminal in line.Terminals:
            # Get the node associated with the Terminal
            connectivity_node = terminal.ConnectivityNode
            # Append the name of the ConnectivityNode (bus) to the result list
            results.append(connectivity_node.name)

# Output the result which contains the names of buses connected to the line 
print(results)

['632', '645']


Example 6: Find the wire position data of phase conductor wires used by line 650632?

In [17]:
results = []
name = "650632"

# Traverse through all ACLineSegment instances in the network graph
for line in network.graph[cim.ACLineSegment].values():
    # Check if the line's name matches the specified name '650632'
    if line.name == name:
        # Check if WireSpacingInfo is associated with the ACLineSegment
        if line.WireSpacingInfo is not None:
            wire_spacing_info = line.WireSpacingInfo
            # Loop through all WirePositions associated with the WireSpacingInfo
            for wire_position in wire_spacing_info.WirePositions:
                # Append the x and y coordinates of the wire position to the results list
                coords = dict()
                coords['sequence'] = wire_position.sequenceNumber
                coords['x'] = wire_position.xCoord
                coords['y'] = wire_position.yCoord
                results.append(coords)

# Output the result which contains the positions of the phase conductor wires used by the line named '650632'
print(results)

[]


Example 7: The thermal rating of cables in the model?

In [18]:
results = []
# Cables are described by the classes ConcentricNeutralCableInfo, TapeShieldCableInfo
all_cable_classes = [cim.ConcentricNeutralCableInfo, cim.TapeShieldCableInfo]
# Iterate through all types of cables
for cim_class in all_cable_classes:
    # Check that the type exists in the model
    if cim_class in network.graph:
        # Loop through all instances of each cable info
        for cable_info in network.graph[cim_class].values():
            cable_results = dict()
            cable_results['name'] = cable_info.name
            cable_results['thermal rating'] = cable_info.ratedCurrent
        results.append(cable_results)

print(results)

[]


Example 8: Identify which lines in the model are underground cables?

In [19]:
results = []
        
# Traverse through all ACLineSegment instances in the network graph
for line in network.graph[cim.ACLineSegment].values():
    # First check if the line's phases use underground cables
    for ac_line_segment_phase in line.ACLineSegmentPhases:
        if ac_line_segment_phase.WireInfo is not None:
            wire_info = ac_line_segment_phase.WireInfo
            # Check if the WireInfo class is one of the underground cable types
            if isinstance(wire_info, cim.ConcentricNeutralCableInfo):
                # Append the line's mRID to the results list
                results.append(line.mRID)
            elif isinstance(wire_info, cim.TapeShieldCableInfo):
                # Append the line's mRID to the results list
                results.append(line.mRID)
        
results = set(results)

# Output the result list which contains the names of lines that are underground cables
print(results)

set()


Example 9: Find the names of lines in the model that are underground cables using insulation made from cross linked polyethylene?

In [20]:
results = []
material = cim.WireInsulationKind.crosslinkedPolyethylene

# Check that the graph contains TapeShieldCableInfo
if cim.TapeShieldCableInfo in network.graph:
    # Traverse through all TapeShieldCableInfo instances in the network graph
    for wire_info in network.graph[cim.TapeShieldCableInfo].values():
        # Check if the insulation material matches 
        if wire_info.insulationMaterial == material:
            # Loop through all ACLineSegmentPhases associated with the TapeShieldCableInfo
            for ac_line_segment_phase in wire_info.ACLineSegmentPhases:
                # Get the ACLineSegment associated with the individual phase
                ac_line_segment = ac_line_segment_phase.ACLineSegment
                if ac_line_segment is not None:
                    # Append the ACLineSegment name to the results list
                    results.append(ac_line_segment_phase.ACLineSegment.name)

# Check that the graph contains ConcentricNeutralCableInfo
if cim.ConcentricNeutralCableInfo in network.graph:
    # Traverse through all ConcentricNeutralCableInfo instances in the network graph
    for wire_info in network.graph[cim.ConcentricNeutralCableInfo].values():
        # Check if the insulation material matches 
        if wire_info.insulationMaterial == material:
            # Loop through all ACLineSegmentPhases associated with the ConcentricNeutralCableInfo
            for ac_line_segment_phase in wire_info.ACLineSegmentPhases:
                # Get the ACLineSegment associated with the individual phase
                ac_line_segment = ac_line_segment_phase.ACLineSegment
                if ac_line_segment is not None:
                    # Append the ACLineSegment name to the results list
                    results.append(ac_line_segment_phase.ACLineSegment.name)
        
# Output the results list which contains the names of lines that are underground cables using the specified insulation material
print(results)

[]


Example 10: What is the radius of the wire used in the phases of line with mRID 6DEF3353-8276-402F-AC8E-3DEF4A396FFE?

In [21]:
# results = []
# line = network.get_object(mRID = '6DEF3353-8276-402F-AC8E-3DEF4A396FFE')


# # Loop through all ACLineSegmentPhases associated with the ACLineSegment
# for ac_line_segment_phase in line.ACLineSegmentPhases:
#     phase = ac_line_segment_phase.phase
#     # Check if WireInfo is associated with the ACLineSegmentPhase
#     if ac_line_segment_phase.WireInfo is not None:
#         wire_info = ac_line_segment_phase.WireInfo
#         # Append the phase, radius, and gmr of the wire to the result list
#         wire_result = dict()
#         wire_result['phase'] = str(phase)
#         wire_result['coreRadius'] = wire_info.coreRadius
#         wire_result['radius'] = wire_info.radius
#         wire_result['gmr'] = wire_info.gmr
#         results.append(wire_result)

# print(results)

Example 11: What is the resistance of line 645646?

In [22]:
results = []
name = '645646'

for line in network.graph[cim.ACLineSegment].values():
    if name in line.name:
        # Get positive sequence impedance:
        if line.r is not None:
            value = dict()
            value['r'] = line.r
            value['r0'] = line.r
            results.append(value)

        # Get per length impedance
        per_length_impedance = line.PerLengthImpedance

        # Get per length sequence impedance
        if isinstance(per_length_impedance, cim.PerLengthSequenceImpedance):
            value = dict()
            value['per_length_r'] = per_length_impedance.r
            value['per_length_r0'] = per_length_impedance.r0
            results.append(value)
        

        # Get per length phase impedance and phase impedance data
        if isinstance(per_length_impedance, cim.PerLengthPhaseImpedance):
            for phase_impedance_data in per_length_impedance.PhaseImpedanceData:
                value = dict()
                value['phase_impedance_r'] = phase_impedance_data.r
                results.append(value)

print(results)

[{'phase_impedance_r': 0.00082257118}, {'phase_impedance_r': 0.00012837529}, {'phase_impedance_r': 0.00082605086}]


Example 12: What are the location xy coordinates of line named 632645?

In [23]:
results = []
# Define the line name we are searching for
name = '632645'

# Iterate through all ACLineSegment instances in the network graph
for line in network.graph[cim.ACLineSegment].values():
    # Check if the line's name contains the target name
    if name in line.name:
        # Get the location of the line
        location = line.Location
        # Iterate through all position points of the line's location
        for position in location.PositionPoints:
            # Append the x and y coordinates to the result list
            results.append(position.xPosition)
            results.append(position.yPosition)

# Result now contains the xy coordinates
print(results)

['200', '250', '100', '250']
